# KLOD 0.6 — Treinamento no Kaggle

Este notebook usa o Qwen3-1.7B atual como professor congelado e treina um adaptador LoRA separado sobre Qwen3-1.7B-Base.

**Hardware:** selecione o acelerador NVIDIA T4x2 nas configurações do Kaggle. O código de treinamento usa uma GPU de 16 GB; não depende de processamento local do celular.

As 48 perguntas iniciais são uma validação do pipeline, não um conjunto suficiente para criar uma IA de domínio geral.


## 1. Ative o acelerador

Nas configurações do Notebook, escolha **Accelerator → GPU T4x2**. A disponibilidade depende da cota da conta.


In [ ]:
!nvidia-smi
import torch
assert torch.cuda.is_available(), 'GPU CUDA não detectada. Ative T4x2 nas configurações do Notebook.'
print('GPU:', torch.cuda.get_device_name(0))
print('VRAM:', round(torch.cuda.get_device_properties(0).total_memory / 1024**3, 2), 'GiB')


## 2. Clone o código e instale as dependências


In [ ]:
%cd /kaggle/working
!rm -rf KLOD-AI
!git clone --depth 1 -b development https://github.com/japaguimarassi-create/Security-lab.git KLOD-AI
%cd /kaggle/working/KLOD-AI
!pip install -e '.[training]'


## 3. Verifique a GPU e os prompts do professor


In [ ]:
!python scripts/training_preflight.py --stage teacher --config configs/training/distillation.yaml


## 4. Gere respostas com o professor

O professor é carregado sem treinamento e gera respostas candidatas. O filtro automático verifica formato, tamanho, cópia da pergunta e repetição. Ele não comprova a verdade das respostas.


In [ ]:
!python scripts/generate_teacher_data.py --config configs/training/distillation.yaml


## 5. Revise as demonstrações

Examine cada pergunta e resposta. Depois, edite `approved_ids` e coloque somente os identificadores que você verificou. Não aprove respostas apenas porque passaram pelo filtro.


In [ ]:
import json
from pathlib import Path
candidate_file = Path('data/generated/teacher_candidates.jsonl')
candidates = [json.loads(line) for line in candidate_file.read_text(encoding='utf-8').splitlines() if line.strip()]
for item in candidates:
    print('\n' + '=' * 90)
    print(item['id'], '|', item['domain'], '|', item['language'], '| quality:', item['quality_passed'])
    print('PERGUNTA:', item['messages'][-2]['content'])
    print('RESPOSTA:', item['messages'][-1]['content'])

approved_ids = []  # Exemplo: ['pt-001', 'pt-002']; inclua somente IDs revisados
approved_count = 0
for item in candidates:
    item['approved'] = item['id'] in approved_ids and item['quality_passed'] is True
    item['review_status'] = 'approved' if item['approved'] else 'pending_or_rejected'
    if item['approved']:
        item['review_notes'] = 'Reviewed in Kaggle Notebook'
        approved_count += 1
candidate_file.write_text(''.join(json.dumps(item, ensure_ascii=False) + '\n' for item in candidates), encoding='utf-8')
print('Approved:', approved_count)
assert approved_count >= 20, 'Revise e aprove pelo menos 20 exemplos antes do treino.'


## 6. Exporte e valide os exemplos aprovados


In [ ]:
!python scripts/approve_teacher_data.py --minimum 20
!python scripts/training_preflight.py --stage sft --config configs/training/distillation_sft.yaml


## 7. Treine o aluno

Configuração inicial: QLoRA NF4, rank 16, alpha 32, dropout 0.05, learning rate 2e-5, sequência de 2048 tokens, batch 1, acumulação 16 e cinco épocas.

O resultado é um adaptador treinado, não um novo checkpoint completo.


In [ ]:
!python scripts/train_sft.py --config configs/training/distillation_sft.yaml


## 8. Compare o aluno com o professor


In [ ]:
!python scripts/evaluate_teacher_student.py --config configs/training/distillation_eval.yaml


## 9. Empacote os resultados\n\nO notebook cria um ZIP somente do adaptador e outro pacote com o adaptador, métricas, exemplos aprovados, configuração e avaliação cega. A chave que revela professor/aluno é mantida fora dos dois pacotes.\n\nAntes de compartilhar a pasta de saída do Kaggle, mantenha a chave de avaliação privada.\n

In [ ]:
from pathlib import Path\nimport shutil\n\nproject = Path('/kaggle/working/KLOD-AI')\noutput_dir = project / 'outputs/klod-teacher-distilled'\nassert output_dir.is_dir(), 'Adaptador não encontrado. O treinamento precisa terminar primeiro.'\n\nadapter_zip = shutil.make_archive('/kaggle/working/klod-teacher-distilled-adapter', 'zip', output_dir)\nrelease_root = Path('/kaggle/working/klod-teacher-distillation-results')\nif release_root.exists():\n    shutil.rmtree(release_root)\nrelease_root.mkdir(parents=True)\nshutil.copytree(output_dir, release_root / 'adapter')\n\nrequired_files = [\n    Path('data/processed/teacher_approved.jsonl'),\n    Path('evaluation/distillation_blind.jsonl'),\n    Path('outputs/klod-teacher-distilled/training_metrics.json'),\n    Path('configs/training/distillation.yaml'),\n    Path('configs/training/distillation_sft.yaml'),\n    Path('configs/training/distillation_eval.yaml'),\n]\nfor relative in required_files:\n    source = project / relative\n    assert source.is_file(), f'Resultado necessário não encontrado: {source}'\n    target = release_root / relative\n    target.parent.mkdir(parents=True, exist_ok=True)\n    shutil.copy2(source, target)\n\n(release_root / 'README.txt').write_text(\n    'KLOD 0.6 teacher-student run\\nAdapter: ./adapter\\n'\n    'Blind evaluation: ./evaluation/distillation_blind.jsonl\\n'\n    'The answer key is intentionally excluded.\\n',\n    encoding='utf-8'\n)\nresults_zip = shutil.make_archive('/kaggle/working/klod-teacher-distillation-results', 'zip', release_root)\nanswer_key = project / 'evaluation/distillation_answer_key.json'\nassert answer_key.is_file(), 'Chave de avaliação não encontrada.'\nprint('Adapter ZIP:', adapter_zip)\nprint('Results ZIP:', results_zip)\nprint('PRIVATE answer key (do not publish):', answer_key)\n

## Observações

- Se a sessão for interrompida, confira se o adaptador parcial foi salvo antes de reiniciar.
- Não inclua a chave da avaliação cega no material de revisão.
- Não use o conjunto de avaliação como treinamento.
- A primeira base de 48 prompts é pequena. Amplie os exemplos revisados antes de declarar que o aluno superou o professor.
- Mantenha o checkpoint e os dados aprovados associados ao commit e à configuração que os produziram.
